### CA2 - Credit Default Risk: Exploratory Data Analysis and Data Preparation
**Module:** Strategic Thinking                                   
**Programme:** Higher Diploma in Data Analytics in Business                            
**Student:** Dashzeveg Purevtogtokh  
**Student ID:** 2026328                         
**Assignment:** CA2                                     
**GitHub Repository:** https://github.com/2026328-Dashzeveg-Purevtogtokh/Strategic-Thinking-Sep26

This notebook continues the CA1 capstone project.  
The main purpose of this CA2 notebook is to:

- understand the credit-card default dataset;
- check data quality;
- perform descriptive statistics;
- perform exploratory data analysis (EDA);
- apply suitable inferential statistics;
- prepare the data for later machine-learning work;
- record findings, limitations and recommendations.

#### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from scipy.stats import chi2_contingency, ttest_ind

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

#### 2. Load the Dataset

In [2]:
from pathlib import Path
import pandas as pd

filename = "default of credit card clients.xls"

paths = [
    Path("data") / filename,
    Path("CA1-Credit-Default-Risk/data") / filename,
]

DATA_PATH = next((p for p in paths if p.is_file()), None)

if DATA_PATH is None:
    raise FileNotFoundError(f"Dataset not found from: {Path.cwd()}")

df_raw = pd.read_excel(DATA_PATH, header=1, engine="calamine")
df = df_raw.copy()

print("Loaded:", DATA_PATH)
print("Rows, columns:", df_raw.shape)
display(df_raw.head())

Loaded: data\default of credit card clients.xls
Rows, columns: (30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


#### 3. Initial Data Understanding

The CA1 project identified this as the UCI **Default of Credit Card Clients** dataset.

The dataset contains historical customer credit and repayment information.  
Important groups of variables include:

- `LIMIT_BAL` – credit limit;
- `SEX`, `EDUCATION`, `MARRIAGE`, `AGE` – customer characteristics;
- `PAY_0` to `PAY_6` – repayment-status variables;
- `BILL_AMT1` to `BILL_AMT6` – bill amounts;
- `PAY_AMT1` to `PAY_AMT6` – previous payment amounts;
- `default payment next month` – target variable.

The next cells confirm the structure of the actual uploaded dataset.

In [3]:
df = df.replace(["none", "None", "NONE", "null", "NULL", "NaN", ""], np.nan)

In [4]:
df.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [5]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

Number of rows: 30000
Number of columns: 25

Column names:
01. ID
02. LIMIT_BAL
03. SEX
04. EDUCATION
05. MARRIAGE
06. AGE
07. PAY_0
08. PAY_2
09. PAY_3
10. PAY_4
11. PAY_5
12. PAY_6
13. BILL_AMT1
14. BILL_AMT2
15. BILL_AMT3
16. BILL_AMT4
17. BILL_AMT5
18. BILL_AMT6
19. PAY_AMT1
20. PAY_AMT2
21. PAY_AMT3
22. PAY_AMT4
23. PAY_AMT5
24. PAY_AMT6
25. default payment next month


In [6]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
print(df.columns.tolist())

['id', 'limit_bal', 'sex', 'education', 'marriage', 'age', 'pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6', 'bill_amt1', 'bill_amt2', 'bill_amt3', 'bill_amt4', 'bill_amt5', 'bill_amt6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6', 'default_payment_next_month']


In [7]:
# Rename target value
df.columns = [col.strip().lower().replace(" ", "_") for col in df.columns]
df = df.rename(columns={"default_payment_next_month": "default"})

In [8]:
df = df.map(lambda s: s.lower() if isinstance(s, str) else s)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   id         30000 non-null  int64
 1   limit_bal  30000 non-null  int64
 2   sex        30000 non-null  int64
 3   education  30000 non-null  int64
 4   marriage   30000 non-null  int64
 5   age        30000 non-null  int64
 6   pay_0      30000 non-null  int64
 7   pay_2      30000 non-null  int64
 8   pay_3      30000 non-null  int64
 9   pay_4      30000 non-null  int64
 10  pay_5      30000 non-null  int64
 11  pay_6      30000 non-null  int64
 12  bill_amt1  30000 non-null  int64
 13  bill_amt2  30000 non-null  int64
 14  bill_amt3  30000 non-null  int64
 15  bill_amt4  30000 non-null  int64
 16  bill_amt5  30000 non-null  int64
 17  bill_amt6  30000 non-null  int64
 18  pay_amt1   30000 non-null  int64
 19  pay_amt2   30000 non-null  int64
 20  pay_amt3   30000 non-null  int64
 21  pay_amt4   30000 non-nu

**My observation:** The dataset contains 30,000 records and 25 columns. All expected variables are present, including customer information, repayment status, bill amounts and previous payment amounts. The original target column, `default payment next month`, was renamed to `default` to make it easier to use in the analysis. All columns are currently stored as integer (`int64`) data types. No unexpected data types were identified. The `id` column is a customer identifier and will not be used as a predictive feature in the later modelling stage.